# FM Vision Encoder Comparison — Segmentation

Benchmarks **11 foundation-model image encoders** with three frozen-feature segmentation heads across chest-X-ray segmentation datasets:

- **`linear_probe_seg`** — single `1×1 Conv2d` + bilinear upsample (~150 params).
- **`conv_probe_seg`** — `1×1 → LN2d → 3×3 → LN2d → 1×1` conv block + bilinear upsample (~0.79M params).
- **`upernet_seg`** — Simple Feature Pyramid + HF `UperNetHead` (PSP + FPN fuse, ~9.8M params).

| Aspect | SIIM-ACR Pneumothorax | Montgomery CXR |
|---|---|---|
| Task | Binary pneumothorax segmentation | Binary lung-mask segmentation |
| Split | 5-fold `GroupKFold` on `patient_id` | 5-fold `GroupKFold` (TB-positive cases only) |
| Variance | `n_folds=5` | `n_folds=5` |
| Foreground prevalence | ~1 % of pixels (sparse) | ~30–40 % (near-balanced) |

**Backbones**:
- Vision-only (6): RAD-DINO, DINOv3, CheXFound, Ark+, Medical-MAE, EVA-X
- Vision-language (5, image-tower only here): BiomedCLIP, CheXagent, MedSigLIP, MedImageInsights, SigLIP2

All backbones are built with `output_keys={"img", "mask"}` so they return dense patch features `Tensor[B, D, H', W']` and the transform wires the mask preprocessor.

**Early stopping** is on by default for all three heads (macro-foreground `dice` on the val signal; restore best-epoch state). In k-fold mode the val signal reuses the held-out fold.

**Verification workflow**: shrink `BACKBONES_TO_RUN` / `DATASETS_TO_RUN` / `PROBES_TO_RUN` to one entry each, step through, confirm CSV schemas. Then expand to the full grid.

## Imports

In [ ]:
import os

# Control which GPU(s) are visible to this notebook.
# Set CUDA_VISIBLE_DEVICES in your shell before launching Jupyter to override,
# e.g.:  CUDA_VISIBLE_DEVICES=2 jupyter lab
# Or change the default value below.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "7")
print(f"CUDA_VISIBLE_DEVICES = {os.environ['CUDA_VISIBLE_DEVICES']}")

In [ ]:
from __future__ import annotations

import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from radharmony.evaluator import (
    LinearProbeSegEvaluator,
    ConvProbeSegEvaluator,
    UPerNetSegEvaluator,
)

## Configuration

Edit the `*_TO_RUN` lists to control what executes. Verification phase: keep them minimal until the schemas check out.

In [ ]:
OUTPUT_DIR  = Path("./eval_out/fm_comparison_seg/")
RESULTS_DIR = OUTPUT_DIR / "results"
# Same convention as notebooks/backbones/done/*: per-(backbone × dataset) subdir under one cache root.
# The MONAI PersistentDataset dir lives here (segmentation evaluator does not cache embeddings).
CACHE_ROOT  = Path("/path/to/cache/")
for d in (OUTPUT_DIR, RESULTS_DIR, CACHE_ROOT):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda:0"

# ─── what to run ──────────────────────────────────────
BACKBONES_TO_RUN = [
    # vision-only
    "rad_dino", "dinov3", "chexfound", "ark_plus", "medical_mae", "eva_x",
    # vision-language (image tower only — segmentation uses dense features, not text)
    "biomed_clip", "chexagent", "medsiglip", "medimageinsights", "siglip2",
]
PROBES_TO_RUN    = ["linear_seg", "conv_probe_seg", "upernet_seg"]  # segmentation heads to run
DATASETS_TO_RUN  = ["siim_acr_ptx", "montgomery_cxr"]  # extend with more seg datasets as builders are wired below
# ────────────────────────────────────────────────────────────

# K-fold protocol (variance comes from the fold axis)
N_FOLDS         = 5
N_TRAIN_SAMPLES = None             # None = full fold train split; [200, 500, 1500] = sweep

# Training
EPOCHS          = 20
LR              = 1e-3
WEIGHT_DECAY    = 0.05

# Early stopping (forwarded into all three seg evaluators)
# - Val signal: both k-fold and fixed-split modes carve a VAL_FRACTION
#   slice from the train subsample. The held-out fold (k-fold) and the
#   test set (fixed-split) are used only for scoring.
# - Set VAL_FRACTION=0.0 to disable early stopping.
EARLY_STOP_METRIC   = "dice"       # any key in the seg metric panel: dice|iou|pixel_acc|tpr|tnr|ppv|npv
EARLY_STOP_PATIENCE = 5
VAL_FRACTION        = 0.1

# Segmentation head
NUM_CLASSES     = 2                # binary (background + foreground)

# Per-dataset class weights: foreground sparsity differs by task.
# SIIM-ACR PTX foreground ≈ 1–3% of pixels → upweight foreground.
# Montgomery lung masks ≈ 30–40% of pixels → near-balanced.
CLASS_WEIGHTS = {
    "siim_acr_ptx":   [1.0, 5.0],
    "montgomery_cxr": [1.0, 1.0],
}

# Per-dataset batch size: smaller corpora can use a smaller batch to fit
# bigger crops in memory; large corpora benefit from larger batches.
# Note: UPerNet's PSP module fails at batch_size=1 (BatchNorm on pool-to-1×1
# branch); keep all entries >=2.
BATCH_SIZE = {
    "siim_acr_ptx":   16,
    "montgomery_cxr": 8,
}

# Common
BASE_SEED       = 0
NUM_WORKERS     = 4
AUTOCAST_DTYPE  = torch.bfloat16
SAVE_PREDICTIONS = False           # set True to dump GT/Pred/Prob PNGs under RESULTS_DIR


## Backbone registry

Each entry is `(factory_callable, factory_kwargs, has_text_encoder)`. For segmentation we always pass `output_keys={"img", "mask"}`; vision-language factories still return their 4-tuple but we only keep the image encoder.

In [ ]:
from radharmony.evaluator.backbones import (
    make_raddino, make_dinov3, make_chexfound, make_ark_plus,
    make_medical_mae, make_eva_x,
    make_biomed_clip, make_chexagent, make_medsiglip,
    make_medimageinsights, make_siglip2,
)

SEG_KWARGS = {"device": DEVICE, "output_keys": {"img", "mask"}}

BACKBONE_FACTORIES: dict = {
    # vision-only
    "rad_dino":         (make_raddino,         SEG_KWARGS, False),
    "dinov3":           (make_dinov3,          SEG_KWARGS, False),
    "chexfound":        (make_chexfound,       SEG_KWARGS, False),
    "ark_plus":         (make_ark_plus,        SEG_KWARGS, False),
    "medical_mae":      (make_medical_mae,     SEG_KWARGS, False),
    "eva_x":            (make_eva_x,           SEG_KWARGS, False),
    # vision-language (we keep only the image encoder for segmentation)
    "biomed_clip":      (make_biomed_clip,     SEG_KWARGS, True),
    "chexagent":        (make_chexagent,       SEG_KWARGS, True),
    "medsiglip":        (make_medsiglip,       SEG_KWARGS, True),
    "medimageinsights": (make_medimageinsights,SEG_KWARGS, True),
    "siglip2":          (make_siglip2,         SEG_KWARGS, True),
}

## Dataset builders

Each builder takes a backbone-specific `transform` (which includes the mask preprocess step) and returns the RadHarmony dataset with `output_mask=True`. NAS paths are inlined; edit for your environment.

To add another segmentation dataset:
1. Add a builder here that returns a RadHarmony dataset with `output_mask=True`.
2. Add its key (e.g. `"ranzcr_clip"`) to `DATASETS_TO_RUN`.
3. Wire it into the `build_dataset` dispatcher below.

In [ ]:
from radharmony.dataset import SIIMACRPTXDataset, MontgomeryCXRDataset
from radharmony.harmonizer import MontgomeryCXRHarmonizer

SIIM_IMAGE_DIR = "/path/to/SIIM_ACR_Pneumothorax/dicom-images-train/"
SIIM_MASK_DIR  = "/path/to/SIIM_ACR_Pneumothorax/masks/"

# Montgomery — fused (left ∪ right) lung mask per image.
# We restrict to TB-positive cases (58 of 138) per study design.
MONT_IMAGE_DIR = "/path/to/Montgomery-CXR/MontgomerySet/CXR_png/"
MONT_MASK_DIR  = "/path/to/mask_verify_out/montgomery_fused/"


def build_siim_acr_ptx(transform, cache_dir=None):
    return SIIMACRPTXDataset(
        base_image_dir=SIIM_IMAGE_DIR,
        mask_output_dir=SIIM_MASK_DIR,
        transform=transform,
        cache_dir=cache_dir,
        output_mask=True,
    )


def build_montgomery_cxr(transform, cache_dir=None):
    """Montgomery CXR — TB-positive cases only (lung-mask segmentation)."""
    h = MontgomeryCXRHarmonizer(base_dir=MONT_IMAGE_DIR)
    df = h.harmonize(mask_output_dir=MONT_MASK_DIR)
    df_pos = df[df["tb"] == 1].reset_index(drop=True)
    print(f"[montgomery_cxr] TB-positive only: {len(df_pos):,} of {len(df):,} cases")
    return MontgomeryCXRDataset(
        base_image_dir=MONT_IMAGE_DIR,
        transform=transform,
        cache_dir=cache_dir,
        output_mask=True,
        harmonized_df=df_pos,
    )

# ── extend here ───────────────────────────────────────────────────────────
# from radharmony.dataset import RANZCRClipDataset
# RANZCR_IMAGE_DIR = "/path/to/ranzcr/train/"
# RANZCR_MASK_DIR  = "/path/to/ranzcr/masks/"
# RANZCR_ANN_CSV   = "/path/to/ranzcr/train_annotations.csv"
# def build_ranzcr_clip(transform, cache_dir=None):
#     return RANZCRClipDataset(
#         base_image_dir=RANZCR_IMAGE_DIR,
#         mask_output_dir=RANZCR_MASK_DIR,
#         annotations_csv_path=RANZCR_ANN_CSV,
#         transform=transform,
#         cache_dir=cache_dir,
#         output_mask=True,
#     )

DATASET_BUILDERS = {
    "siim_acr_ptx":   build_siim_acr_ptx,
    "montgomery_cxr": build_montgomery_cxr,
    # "ranzcr_clip": build_ranzcr_clip,
}

## Probe runners

Each `run_*_seg` issues a **k-fold** call using `GroupKFold(n_folds)` on `patient_id`. The frozen encoder is re-applied per batch (no embedding cache — segmentation needs dense per-pixel features, which are too large to cache to disk like a single CLS token).

The three runners share `_COMMON_SEG_KWARGS` so the early-stopping knobs (`EARLY_STOP_METRIC`, `EARLY_STOP_PATIENCE`, `VAL_FRACTION`) and training hyper-params are forwarded to all heads identically.

In [ ]:
def _has_result(dataset_key, probe_key, backbone_key):
    return (RESULTS_DIR / dataset_key / probe_key / backbone_key / "results.csv").exists()


def _output_dir(dataset_key, probe_key, backbone_key):
    p = RESULTS_DIR / dataset_key / probe_key / backbone_key
    p.mkdir(parents=True, exist_ok=True)
    return str(p)


def _monai_cache(backbone_key, dataset_key):
    return str(CACHE_ROOT / f"{backbone_key}_{dataset_key}")

In [ ]:
_COMMON_SEG_KWARGS = dict(
    num_classes=NUM_CLASSES,
    n_folds=N_FOLDS,
    n_train_samples=N_TRAIN_SAMPLES,
    epochs=EPOCHS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    save_predictions=SAVE_PREDICTIONS,
    num_workers=NUM_WORKERS,
    device=DEVICE,
    autocast_dtype=AUTOCAST_DTYPE,
    base_seed=BASE_SEED,
    early_stop_metric=EARLY_STOP_METRIC,
    early_stop_patience=EARLY_STOP_PATIENCE,
    val_fraction=VAL_FRACTION,
)


def _run_seg(cls, probe_key, image_encoder, dataset, *, backbone_key, dataset_key, **extra):
    ev = cls(
        image_encoder=image_encoder,
        dataset=dataset,
        class_weights=CLASS_WEIGHTS[dataset_key],
        batch_size=BATCH_SIZE[dataset_key],
        output_dir=_output_dir(dataset_key, probe_key, backbone_key),
        **_COMMON_SEG_KWARGS,
        **extra,
    )
    df = ev.evaluate()
    ev.save_results(df)
    return df


def run_linear_seg(image_encoder, dataset, *, backbone_key, dataset_key):
    return _run_seg(LinearProbeSegEvaluator, "linear_seg",
                    image_encoder, dataset,
                    backbone_key=backbone_key, dataset_key=dataset_key)


def run_conv_probe_seg(image_encoder, dataset, *, backbone_key, dataset_key):
    return _run_seg(ConvProbeSegEvaluator, "conv_probe_seg",
                    image_encoder, dataset,
                    backbone_key=backbone_key, dataset_key=dataset_key)


def run_upernet_seg(image_encoder, dataset, *, backbone_key, dataset_key):
    return _run_seg(UPerNetSegEvaluator, "upernet_seg",
                    image_encoder, dataset,
                    backbone_key=backbone_key, dataset_key=dataset_key)


PROBE_RUNNERS = {
    "linear_seg":     run_linear_seg,
    "conv_probe_seg": run_conv_probe_seg,
    "upernet_seg":    run_upernet_seg,
}


## Main loop — per backbone

For each backbone in `BACKBONES_TO_RUN`:
1. Build encoder + transform (with `output_keys={"img", "mask"}`) via factory.
2. Build each segmentation dataset with that transform.
3. Run the linear-probe segmentation evaluator (k-fold).
4. Tear down the encoder and clear CUDA cache before the next backbone.

Per (dataset × backbone) is resume-safe — if `results.csv` exists, the cell skips it.

In [ ]:
for backbone_key in BACKBONES_TO_RUN:
    factory, kwargs, has_text = BACKBONE_FACTORIES[backbone_key]
    print(f"\n[backbone] {backbone_key}")

    if has_text:
        transform, encoder, *_ = factory(**kwargs)
    else:
        transform, encoder = factory(**kwargs)

    for dataset_key in DATASETS_TO_RUN:
        if dataset_key not in DATASET_BUILDERS:
            print(f"  skip {dataset_key}: no builder registered")
            continue
        ds = None
        for probe_key in PROBES_TO_RUN:
            if _has_result(dataset_key, probe_key, backbone_key):
                print(f"  skip {dataset_key}/{probe_key}: results.csv exists")
                continue
            if ds is None:
                ds = DATASET_BUILDERS[dataset_key](
                    transform, cache_dir=_monai_cache(backbone_key, dataset_key),
                )
            print(f"  running {dataset_key}/{probe_key} ...")
            PROBE_RUNNERS[probe_key](encoder, ds, backbone_key=backbone_key, dataset_key=dataset_key)
        if ds is not None:
            del ds

    # release GPU before next backbone
    del encoder
    gc.collect()
    torch.cuda.empty_cache()

## Aggregate

Walk `RESULTS_DIR`, concatenate every `results.csv`, tag rows with their `{dataset, evaluator, backbone}`, and write `all_results.csv`.

In [ ]:
records = []
for csv in RESULTS_DIR.glob("*/*/*/results.csv"):
    parts = csv.relative_to(RESULTS_DIR).parts  # (dataset, evaluator, backbone, "results.csv")
    df = pd.read_csv(csv)
    df["dataset"]   = parts[0]
    df["evaluator"] = parts[1]
    df["backbone"]  = parts[2]
    records.append(df)

all_df = pd.concat(records, ignore_index=True) if records else pd.DataFrame()
all_df.to_csv(OUTPUT_DIR / "all_results.csv", index=False)
print(f"Wrote {len(all_df):,} rows to {OUTPUT_DIR / 'all_results.csv'}")
all_df.head()